# Итоговый проект DWH — разбор решения

Этот ноутбук дополняет основной проект: **Kafka + ClickHouse + RustFS + Spark + Airflow**. Он предназначен для изучения договора входных данных и независимой проверки результатов, **а не для замены DAG**. Полное выполнение запускается через `docker-compose.yaml` по инструкции `README_IMPLEMENTATION.md`.

Сохранённые выводы ниже получены независимым Python-расчётом по исходным JSONL преподавателя. Это **контрольные числа, не результаты запуска Spark**.

## 1. Откуда берутся данные

В каталоге `data/fixture` находятся 28 сообщений о событиях (26 уникальных) и 9 сообщений об изменениях профиля (8 уникальных). В `data/demo` — 6 138 сообщений (6 000 уникальных событий) и 203 изменения профилей (202 уникальных). Сообщения повторяются по ID, но `available_at` у повторной доставки может отличаться.

`event_time` — время события; `effective_at` — время начала действия профиля. Поле `available_at` означает только момент поступления в Kafka и **не используется как ключ исторического JOIN**.

In [1]:
from pathlib import Path
import sys, json

# Ячейки работают как из корня проекта, так и из папки notebooks/.
PROJECT = Path.cwd() if (Path.cwd() / "data" / "fixture").exists() else Path.cwd().parent
assert (PROJECT / "data" / "fixture" / "manifest.json").exists()
sys.path.insert(0, str(PROJECT / "scripts"))
from reference import compute

for ds in ("fixture", "demo"):
    manifest = json.loads((PROJECT / "data" / ds / "manifest.json").read_text(encoding="utf-8"))
    print(ds, ":", manifest["unique_events"], "уникальных событий,",
          manifest["users"], "пользователей")

fixture : 26 уникальных событий, 3 пользователей
demo : 6000 уникальных событий, 100 пользователей


## 2. Устранение повторных доставок

Повторная Kafka-доставка содержит тот же `event_id`. До расчёта просмотров и слов оставляем **одно событие на ID**, иначе число просмотров и длительность будут завышены. В Spark это реализовано с помощью оконной функции `row_number()` по `event_id` с сортировкой по `available_at`. Архив S3 при этом хранит все исходные сообщения без удаления повторов.

In [2]:
result = compute(PROJECT / "data" / "fixture")
print("Уникальных событий:", result["events"])
print("Просмотров:", sum(x[0] for x in result["minute"].values()))
print("Просмотрено секунд:", sum(x[1] for x in result["minute"].values()))

Уникальных событий: 26
Просмотров: 20
Просмотрено секунд: 3141


## 3. История изменений профиля (SCD2)

Для каждого пользователя записи о стране и тарифе упорядочиваются по `effective_at`. Непосредственно повторяющееся состояние не создаёт новую версию, но **возврат** к прежнему состоянию после промежуточного изменения создаёт. Конец версии равен началу следующей; интервалы имеют вид **`[valid_from, valid_to)`**.

В Spark это выражено через `lag()` для обнаружения изменений и `lead()` для вычисления `valid_to`. Код: `jobs/pipeline.py` → `build_scd2`.

In [3]:
for ds in ("fixture", "demo"):
    result = compute(PROJECT / "data" / ds)
    print(f"{ds}: {result['users']} пользователей, {result['scd2_rows']} исторических версий")

fixture: 3 пользователей, 7 исторических версий
demo: 100 пользователей, 187 исторических версий


## 4. Историческое соединение просмотров

Для каждого уникального `view` должны существовать **ровно одна** строка каталога (`content_id`) и **ровно одна** версия профиля с `valid_from <= event_time` и `(valid_to IS NULL OR event_time < valid_to)`. Здесь используется **время просмотра**, а не время поступления Kafka и не нынешнее состояние профиля.

Это условие проверяется перед расчётом и повторно в `validate_and_compare_rdd`.

**Важно:** `content_id = 0` — допустимое значение, его нельзя отфильтровывать как «пустой идентификатор».

In [4]:
result = compute(PROJECT / "data" / "fixture")
print("Просмотр content_id=0 в первую минуту:", result["minute"][("2026-01-01T00:01:00", 0)])
print("Топ-1 за 1 января (content_id, просмотры, секунды):", result["top5"][("2026-01-01", 1)])

Просмотр content_id=0 в первую минуту: [1, 60]
Топ-1 за 1 января (content_id, просмотры, секунды): (0, 3, 317)


## 5. Какие витрины считаются

1. Поминутные просмотры и секунды — ключ `minute + content_id`.
2. Дневные просмотры по стране и тарифу — исторические значения на дату каждого просмотра.
3. Топ-5 контента за день — по количеству просмотров, при равенстве сначала меньший `content_id`.
4. Частоты слов из уникальных поисковых событий — нижний регистр, разбиение по пробельным символам, слова длиннее четырёх букв. Повтор слова внутри запроса считается повторно.

Функции расчёта находятся в `jobs/pipeline.py`. Топ-5 и частоты слов повторно вычисляются через RDD и сравниваются с DataFrame.


In [5]:
for ds in ("fixture", "demo"):
    x = compute(PROJECT / "data" / ds)
    print(f"{ds}: minutes={len(x['minute'])}, country/tier={len(x['daily'])}, "
          f"top5={len(x['top5'])}, search_words={len(x['words'])}")

fixture: minutes=20, country/tier=4, top5=12, search_words=11
demo: minutes=4243, country/tier=36, top5=15, search_words=39


## 6. Поступление поздних сообщений

Готовые датасеты воспроизводятся в два этапа по `available_at`: до `2026-01-04T00:00:00Z` (не включая границу) и начиная с этой даты. После второй отправки Spark должен повторно рассчитать затронутые дни. Временные интервалы профиля определяются по `effective_at`, поэтому опоздавшее изменение может поменять результаты за прошлые сутки.

Проверить оба этапа и повторный запуск можно по `README_IMPLEMENTATION.md`. Этот ноутбук не подменяет результаты Airflow.


## 7. Совмещение оперативного и пакетного слоёв

В ClickHouse отчёт `final_minute` использует одну и ту же детализацию и показатели:

- `minute < T` → только опубликованные данные Spark;
- `minute >= T` → только оперативные события из Kafka, с дедупликацией по `event_id`.

Только после проверки **всех четырёх** витрин нового дня активируется `publish_id`. Лишь после этого граница `T` может сдвинуться через непрерывно опубликованные дни. Это позволяет избегать разрывов и двойного учёта.

Найти реализацию можно в `sql/schema.sql` (`final_minute`) и `jobs/pipeline.py` (`publish_marts`).

## 8. Проверка результата и сдача

После запуска всего проекта и повторной обработки поздних сообщений выполните:

```powershell
docker compose run --rm tools python /opt/airflow/scripts/check_results.py --dataset fixture
```

Ожидаемый признак успешной проверки — `ALL INDEPENDENT CHECKS PASSED`. Затем сохраните скриншоты DAG, ClickHouse и RustFS в отчёт. **Не выдавайте приведённые выше контрольные числа за самостоятельно запущенные Spark-измерения**: реальные выводы должны быть получены в вашем окружении.
